# Reachy Mini playground

Interactive notebook for the simulated (or real) Reachy Mini Lite.

**Before you start:** start the simulator (double-click `2-START-SIMULATOR.command`, or the VS Code task *Reachy: start simulator*) and pick the kernel **Python (Reachy Mini)**.

The robot is controlled through the daemon on `localhost:8000`. The same cells work on a real Reachy Mini Lite connected by USB, once its daemon is running instead of the sim.

## 1. Connect

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from reachy_mini import ReachyMini
from reachy_mini.utils import create_head_pose

# One persistent connection for the whole notebook (run the last cell to close it).
# no_media: motion only. Use ReachyMini() if you want the camera.
mini = ReachyMini(media_backend="no_media")
print("connected")

## 2. Basic moves

`goto_target` interpolates smoothly. Head poses are 4x4 matrices built with `create_head_pose` (x/y/z, roll/pitch/yaw). Antennas are `[right, left]` in radians. Interpolation methods: `linear`, `minjerk`, `ease_in_out`, `cartoon`.

In [ ]:
mini.goto_target(head=create_head_pose(z=15, roll=15, mm=True, degrees=True), antennas=[0.5, -0.5], duration=1.0)
mini.goto_target(head=create_head_pose(yaw=30, degrees=True), body_yaw=np.deg2rad(20), duration=1.0, method="ease_in_out")
mini.goto_target(head=create_head_pose(), antennas=[0, 0], body_yaw=0.0, duration=1.0)

## 3. Live sliders

Move the sliders and watch the MuJoCo window. Requires `ipywidgets` (installed by setup). Out-of-range values are clamped by the SDK (pitch/roll +-40 deg, body yaw +-160 deg, max 65 deg between head and body yaw).

In [ ]:
import ipywidgets as w

def drive(x=0.0, y=0.0, z=0.0, roll=0.0, pitch=0.0, yaw=0.0, antenna_r=0.0, antenna_l=0.0, body_yaw=0.0):
    mini.set_target(
        head=create_head_pose(x=x, y=y, z=z, roll=roll, pitch=pitch, yaw=yaw, mm=True, degrees=True),
        antennas=np.deg2rad([antenna_r, antenna_l]),
        body_yaw=float(np.deg2rad(body_yaw)),
    )

mm = dict(min=-20.0, max=20.0, step=1.0)
deg = dict(min=-40.0, max=40.0, step=1.0)
w.interact(
    drive,
    x=w.FloatSlider(**mm), y=w.FloatSlider(**mm), z=w.FloatSlider(**mm),
    roll=w.FloatSlider(**deg), pitch=w.FloatSlider(**deg),
    yaw=w.FloatSlider(min=-90.0, max=90.0, step=1.0),
    antenna_r=w.FloatSlider(min=-90.0, max=90.0, step=1.0),
    antenna_l=w.FloatSlider(min=-90.0, max=90.0, step=1.0),
    body_yaw=w.FloatSlider(min=-90.0, max=90.0, step=1.0),
);

## 4. Record and plot a motion

Run a generated trajectory with `set_target` at 50 Hz and log what the robot actually did (`get_current_joint_positions`).

In [ ]:
rate, seconds = 50, 6
log_t, log_cmd, log_ant = [], [], []
t0 = time.perf_counter()
for i in range(rate * seconds):
    t = time.perf_counter() - t0
    yaw = 30 * np.sin(2 * np.pi * 0.3 * t)
    a = np.deg2rad(40 * np.sin(2 * np.pi * 0.6 * t))
    mini.set_target(head=create_head_pose(yaw=yaw, degrees=True), antennas=[a, -a])
    _, antennas = mini.get_current_joint_positions()
    log_t.append(t); log_cmd.append(a); log_ant.append(antennas[0])
    time.sleep(max(0.0, (i + 1) / rate - (time.perf_counter() - t0)))

plt.figure(figsize=(8, 3))
plt.plot(log_t, np.rad2deg(log_cmd), label="commanded right antenna")
plt.plot(log_t, np.rad2deg(log_ant), label="measured right antenna")
plt.xlabel("time [s]"); plt.ylabel("deg"); plt.legend(); plt.tight_layout(); plt.show()
mini.goto_target(head=create_head_pose(), antennas=[0, 0], duration=1.0)

## 5. Recorded emotions from the Hub

Pollen publishes libraries of recorded moves as Hugging Face datasets (first run downloads them). In a notebook use `await mini.async_play_move(...)`, because the sync `play_move` cannot run inside Jupyter's event loop.

In [ ]:
from reachy_mini.motion.recorded_move import RecordedMoves

emotions = RecordedMoves("pollen-robotics/reachy-mini-emotions-library")
print(sorted(emotions.list_moves()))

In [ ]:
for name in ["curious1", "welcoming1"]:
    move = emotions.get(name)
    print(name, "-", move.description)
    await mini.async_play_move(move, initial_goto_duration=1.0, sound=False)

## 6. Close the connection

In [ ]:
mini.goto_target(head=create_head_pose(), antennas=[0, 0], body_yaw=0.0, duration=1.0)
mini.__exit__(None, None, None)  # same cleanup as leaving a `with ReachyMini()` block
print("disconnected")